In [1]:
import numpy as np
import torch
import pandas as pd
import torch.nn.functional as F
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn as nn
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.utils import negative_sampling
import random

import pickle
from tqdm import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'


/home/ivanc/projects/gnn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("../data/dicts/bert_embeddings/protein_esm_35M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)
with open("../data/dicts/bert_embeddings/sm_chemberta_10M_MTR.pkl", 'rb') as f:
    emb_sm = pickle.load(f)
with open("../data/dicts/bert_embeddings/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)
with open("../data/dicts/bert_embeddings/dna_full.pkl", 'rb') as f:
    emb_dna = pickle.load(f)

rawid2enb = emb_prot | emb_sm | emb_rna | emb_dna

In [ ]:
class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        # Использование (-1, -1) позволяет ленивую инициализацию размерностей
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = nn.PReLU(x)
        x = self.conv2(x, edge_index)
        return x

# Декодер: предсказывает вероятность связи на основе эмбеддингов узлов
class EdgeDecoder(torch.nn.Module):
    def forward(self, z_dict, edge_label_index, edge_type):
        # Извлекаем типы источника и цели из типа ребра
        src_type, _, dst_type = edge_type

        # Получаем эмбеддинги для узлов, участвующих в проверяемых ребрах
        z_src = z_dict[src_type][edge_label_index[0]]
        z_dst = z_dict[dst_type][edge_label_index[1]]

        # Скалярное произведение для оценки вероятности (можно заменить на MLP)
        return (z_src * z_dst).sum(dim=-1)

class HeteroLinkPredictionModel(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()
        # to_hetero автоматически адаптирует гомогенный энкодер под гетерограф
        self.encoder = to_hetero(GNNEncoder(hidden_channels, out_channels), metadata, aggr='sum')
        self.decoder = EdgeDecoder()

    def forward(self, x_dict, edge_index_dict, edge_label_index, edge_type):
        # 1. Получаем эмбеддинги для всех узлов
        z_dict = self.encoder(x_dict, edge_index_dict)
        # 2. Предсказываем вероятности для конкретных пар
        return self.decoder(z_dict, edge_label_index, edge_type)

In [2]:
node_types = ['AA', 'DNA', 'RNA', 'SmallMolecule']

rawid_to_local = {}
raw_tensors = {}

for t_name in node_types:
    # Читаем ТОЛЬКО колонку с ID, чтобы узнать тип узлов
    df_nodes = pd.read_csv(f'../data/nodes/nodes_for_mdm/{t_name}.csv', usecols=['id_entity'])

    embeddings_list = []

    for local_idx, raw_id in enumerate(df_nodes['id_entity']):
        # Заполняем маппинги для DataLoader
        rawid_to_local[raw_id] = local_idx

        # Достаем готовый эмбеддинг из твоего словаря
        #emb = rawid2enb[raw_id]

        #embeddings_list.append(emb)

    # Склеиваем список тензоров в одну матрицу для этого типа
    # Получится тензор размерности [N_nodes_of_this_type, embedding_dim]
    #raw_tensors[str(t_name)] = torch.stack(embeddings_list)

df_edges = pd.read_csv('../data/edges/clean_edges_without_NaNm.csv')


df_edges['id_entity_1'] = df_edges['id_entity_1'].apply(lambda x: rawid_to_local[x])
df_edges['id_entity_2'] = df_edges['id_entity_2'].apply(lambda x: rawid_to_local[x])



In [3]:
data = HeteroData()
# for k, v in raw_tensors.items():
#     data[k].x = v.float()

data['AA'].x = torch.randn((70937, 480))
data['DNA'].x = torch.randn((635, 768))
data['RNA'].x = torch.randn((738, 512))
data['SmallMolecule'].x = torch.randn((1001342, 384))

unique_pairs = df_edges[['type_entity_1', 'predicate', 'type_entity_2']].drop_duplicates()

for row in unique_pairs.itertuples(index=False):
    t1 = row.type_entity_1
    r = row.predicate
    t2 = row.type_entity_2

    # Фильтруем основной DataFrame по обоим условиям сразу
    filtered_df = df_edges[(df_edges['type_entity_1'] == t1) &
                           (df_edges['predicate'] == r) &
                           (df_edges['type_entity_2'] == t2)]

    arr = filtered_df[['id_entity_1', 'id_entity_2']].to_numpy().T
    data[t1, r, t2].edge_index = torch.tensor(arr, dtype=torch.long)

transform = T.ToUndirected(merge=False)
data = transform(data)

In [4]:
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    neg_sampling_ratio=0,
    add_negative_train_samples=False,
    edge_types=[
        ('RNA', 'interacts_with', 'AA'),
        ('DNA', 'interacts_with', 'AA'),
        ('DNA', 'interacts_with', 'SmallMolecule'),
        ('RNA', 'interacts_with', 'SmallMolecule'),
        ('AA', 'interacts_with', 'SmallMolecule'),
        ('AA', 'interacts_with', 'AA')
    ],
    rev_edge_types=[
        ('AA', 'rev_interacts_with', 'RNA'),
        ('AA', 'rev_interacts_with', 'DNA'),
        ('SmallMolecule', 'rev_interacts_with', 'DNA'),
        ('SmallMolecule', 'rev_interacts_with', 'RNA'),
        ('SmallMolecule', 'rev_interacts_with', 'AA'),
        ('AA', 'rev_interacts_with', 'AA')
    ],
)

train_data, val_data, test_data = transform(data)

In [6]:
# 1. Базовая модель GraphSAGE (будет конвертирована в гетерогенную)
class BaseGraphSAGE(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        # Использование (-1, -1) позволяет PyG автоматически вывести размерности
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)
        self.prelu = torch.nn.PReLU()

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.prelu(x)
        return self.conv2(x, edge_index)

# Декодер скоров для конкретного типа связи (Inner Product)
def decode(z_dict, edge_index, edge_type):
    src_type, rel_type, dst_type = edge_type
    src_z = z_dict[src_type][edge_index[0]]
    dst_z = z_dict[dst_type][edge_index[1]]
    return (src_z * dst_z).sum(dim=-1)


# def train_epoch(hetero_model, optimizer, criterion, train_loaders, device, accumulation_steps=4):
#     hetero_model.train()
#     total_loss = 0
#     total_batches = 0
#
#     # Создаем итераторы
#     iterators = {etype: iter(loader) for etype, loader in train_loaders.items()}
#     active_types = list(train_loaders.keys())
#
#     # Определяем общее количество шагов по САМОМУ БОЛЬШОМУ лоадеру,
#     # чтобы все типы связей «успели» поучаствовать пропорционально.
#     steps_per_epoch = max(len(l) for l in train_loaders.values())
#
#     pbar = tqdm(total=steps_per_epoch, desc="Balanced Training")
#
#     for _ in range(steps_per_epoch):
#         # Взвешенный выбор: здесь веса равны, чтобы каждый тип связи
#         # встречался одинаково часто (балансировка)
#         edge_type = random.choice(active_types)
#
#         try:
#             batch = next(iterators[edge_type])
#         except StopIteration:
#             # Если редкий тип связи закончился — запускаем его заново (Oversampling)
#             iterators[edge_type] = iter(train_loaders[edge_type])
#             batch = next(iterators[edge_type])
#
#         # ... далее стандартный код обучения (forward, loss, backward, step) ...
#         batch = batch.to(device)
#         optimizer.zero_grad()
#
#         z_dict = hetero_model(batch.x_dict, batch.edge_index_dict)
#         # (ваш код извлечения pos/neg ребер и расчет loss)
#         # Извлекаем целевые ребра для текущего типа связи
#         edges = batch[edge_type].edge_label_index
#         labels = batch[edge_type].edge_label
#
#         # Разделяем на позитивные и негативные
#         pos_edges = edges[:, labels == 1.0]
#         neg_edges = edges[:, labels == 0.0]
#
#         # Синхронизируем размерности
#         min_len = min(pos_edges.size(1), neg_edges.size(1))
#         if min_len == 0:
#             continue
#
#         pos_edges = pos_edges[:, :min_len]
#         neg_edges = neg_edges[:, :min_len]
#
#         # Вычисляем скоры
#         pos_scores = decode(z_dict, pos_edges, edge_type)
#         neg_scores = decode(z_dict, neg_edges, edge_type)
#
#         # Margin Loss
#         target = torch.ones_like(pos_scores)
#         loss = criterion(pos_scores, neg_scores, target)
#
#         loss.backward()
#
#         optimizer.step()
#         optimizer.zero_grad()
#
#
#         total_loss += loss.item()
#
#         total_batches += 1
#         pbar.update(1)
#
#     pbar.close()
#     return total_loss / total_batches


def train_epoch(hetero_model, optimizer, criterion, train_loaders, device):
    hetero_model.train()
    total_loss = 0
    total_batches = 0

    # Последовательно обучаем на батчах каждого типа связи
    for edge_type, loader in train_loaders.items():
        for batch in tqdm(loader):
            batch = batch.to(device)
            optimizer.zero_grad()

            # Получаем эмбеддинги для текущего подграфа
            z_dict = hetero_model(batch.x_dict, batch.edge_index_dict)

            # Извлекаем целевые ребра для текущего типа связи
            edges = batch[edge_type].edge_label_index
            labels = batch[edge_type].edge_label

            # Разделяем на позитивные и негативные
            pos_edges = edges[:, labels == 1.0]
            neg_edges = edges[:, labels == 0.0]

            # Синхронизируем размерности
            min_len = min(pos_edges.size(1), neg_edges.size(1))
            if min_len == 0:
                continue

            pos_edges = pos_edges[:, :min_len]
            neg_edges = neg_edges[:, :min_len]

            # Вычисляем скоры
            pos_scores = decode(z_dict, pos_edges, edge_type)
            neg_scores = decode(z_dict, neg_edges, edge_type)

            # Margin Loss
            target = torch.ones_like(pos_scores)
            loss = criterion(pos_scores, neg_scores, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

    return total_loss / total_batches

In [7]:
target_edge_types = [('RNA', 'interacts_with', 'AA'),
        ('DNA', 'interacts_with', 'AA'),
        ('DNA', 'interacts_with', 'SmallMolecule'),
        ('RNA', 'interacts_with', 'SmallMolecule'),
        ('AA', 'interacts_with', 'SmallMolecule'),
        ('AA', 'interacts_with', 'AA')]

train_loaders = {}
for edge_type in target_edge_types:
    train_loaders[edge_type] = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[25, 10],
        edge_label_index=(edge_type, train_data[edge_type].edge_index),
        neg_sampling_ratio=1.0,
        batch_size=4096,
        subgraph_type = "bidirectional",
        shuffle=True,
    )



In [8]:
# Предполагается, что объект data (HeteroData) уже существует
hetero_model = to_hetero(BaseGraphSAGE(hidden_channels=32), data.metadata(), aggr='sum').to(device)
optimizer = torch.optim.Adam(hetero_model.parameters(), lr=0.001)
criterion = nn.MarginRankingLoss(margin=1.0)

In [14]:
for _ in range(5):
    loss = train_epoch(hetero_model, optimizer, criterion, train_loaders, device)
    print(loss)


100%|██████████| 1/1 [00:00<00:00, 16.71it/s]

100%|██████████| 1/1 [00:00<00:00, 18.82it/s]

100%|██████████| 1/1 [00:00<00:00, 43.02it/s]

100%|██████████| 1/1 [00:00<00:00, 22.89it/s]

100%|██████████| 161/161 [00:38<00:00,  4.16it/s]

100%|██████████| 410/410 [01:12<00:00,  5.62it/s]


0.43286770735745844



100%|██████████| 1/1 [00:00<00:00, 18.68it/s]

100%|██████████| 1/1 [00:00<00:00, 21.41it/s]

100%|██████████| 1/1 [00:00<00:00, 49.78it/s]

100%|██████████| 1/1 [00:00<00:00, 29.99it/s]

100%|██████████| 161/161 [00:39<00:00,  4.08it/s]

100%|██████████| 410/410 [01:14<00:00,  5.53it/s]


0.09363393545474695



100%|██████████| 1/1 [00:00<00:00, 21.53it/s]

100%|██████████| 1/1 [00:00<00:00, 19.81it/s]

100%|██████████| 1/1 [00:00<00:00, 47.64it/s]

100%|██████████| 1/1 [00:00<00:00, 32.80it/s]

 20%|██        | 33/161 [00:08<00:33,  3.85it/s]


KeyboardInterrupt: 

In [9]:
import torch

@torch.no_grad()
def evaluate_kge_style(model, data, edge_type, batch_size=512):
    model.eval()

    # 1. Получаем эмбеддинги для всех типов узлов
    # z_dict будет содержать тензоры для каждого node_type
    z_dict = model(data.x_dict, data.edge_index_dict)

    src_type, rel_type, dst_type = edge_type

    # Ребра, которые мы хотим проверить (тестовые)
    # Предполагаем, что они лежат в data[edge_type].edge_label_index
    test_edges = data[edge_type].edge_label_index

    # Все возможные эмбеддинги "хвостов" (кандидаты для замены)
    all_target_nodes = z_dict[dst_type] # Shape: [N_dst, hidden_channels]
    num_entities = all_target_nodes.size(0)

    ranks = []

    # Итерируемся по тестовым ребрам батчами, чтобы не переполнить память
    for i in range(0, test_edges.size(1), batch_size):
        heads = test_edges[0, i:i + batch_size] # Индексы голов
        tails = test_edges[1, i:i + batch_size] # Индексы истинных хвостов

        # Эмбеддинги голов для текущего батча: [batch_size, hidden_channels]
        h_emb = z_dict[src_type][heads]
        # Эмбеддинги истинных хвостов: [batch_size, hidden_channels]
        t_emb = z_dict[dst_type][tails]

        # Считаем скор для правильных пар: (batch_size,)
        pos_score = (h_emb * t_emb).sum(dim=-1, keepdim=True)

        # Считаем скоры для ВСЕХ возможных хвостов:
        # [batch_size, 1, hidden] * [1, N_dst, hidden] -> [batch_size, N_dst]
        all_scores = torch.matmul(h_emb, all_target_nodes.t())

        # Ранг: сколько скоров больше, чем у правильной пары
        # (all_scores > pos_score).sum(dim=1) дает количество сущностей со скором выше
        # Прибавляем 1, так как лучший ранг — это 1
        rank = (all_scores > pos_score).sum(dim=1) + 1
        ranks.append(rank)

    ranks = torch.cat(ranks).float()

    # Метрики
    mrr = (1.0 / ranks).mean().item()
    hits_10 = (ranks <= 10).float().mean().item()
    hits_1 = (ranks <= 1).float().mean().item()

    return {
        'MRR': mrr,
        'Hits@1': hits_1,
        'Hits@10': hits_10
    }


def evaluate_all_types(model, data, batch_size=32):
    all_results = {}

    # data.edge_types возвращает список всех кортежей (src, rel, dst)
    for edge_type in data.edge_types:
        # Проверяем, есть ли тестовые ребра для этого типа
        # Обычно они хранятся в edge_label_index после сплита

        res = evaluate_kge_style(model, data, edge_type, batch_size)
        all_results[str(edge_type)] = res

    # Считаем среднее (Global Mean) по всем типам связей
    if all_results:
        avg_mrr = sum(r['MRR'] for r in all_results.values()) / len(all_results)
        print(f"\nOverall Mean MRR: {avg_mrr:.4f}")

    return all_results

# Запуск
data = data.to(device)
stats = evaluate_all_types(hetero_model, data)

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.87 GiB. GPU 0 has a total capacity of 7.61 GiB of which 2.20 GiB is free. Including non-PyTorch memory, this process has 4.85 GiB memory in use. Of the allocated memory 4.61 GiB is allocated by PyTorch, and 130.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)